# **R/S BENCHMARK - DATASET GENERATION**

## **1. State limit function**

$$g = k(t) \cdot \frac{R}{z_1} - S \cdot z_2$$

$z_1$ and $z_2$ are normal latent multipliers (mean 1.0, sd 0.028 and 0.096); $k(t) = 1 +
(k_{final}-1)\,t/100$ is the degradation factor.

## **2. Libraries**

In [1]:
import sys
import time
from pathlib import Path

# functions.py sits one directory up
sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd

from functions import *
from UQpy.distributions import Normal, JointIndependent

/home/casa-wand/steam2tb/2024-1_victor_hugo_renata_maria/.venv/lib/python3.11/site-packages/UQpy/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## **3. Random variables and fixed parameters**

Design variables $R$ and $S$, plus everything the emulator needs that isn't a design variable.

In [2]:
r_mean = 5.0   # resistance mean
r_std  = 0.8   # resistance standard deviation
s_mean = 2.0   # load mean
s_std  = 0.6   # load standard deviation

n_samples            = 2000     # Number of design samples
n_latent_samples     = 5000     # Number of latent samples per design sample. Also the filename prefix
n_samples_validation = 500      # Number of validation samples, redrawn at every time step
n_lambdas            = 4        # Number of λs (λ1, λ2, λ3, λ4)
k_factor_final       = 0.3      # Degradation factor at t = 100. Use 1.0 for no time effect
z1_std               = 0.028    # Standard deviation of the resistance latent multiplier
z2_std               = 0.096    # Standard deviation of the load latent multiplier

times = np.linspace(0, 150, 10, endpoint=True)  # Time points for the degradation factor
times

array([  0.        ,  16.66666667,  33.33333333,  50.        ,
        66.66666667,  83.33333333, 100.        , 116.66666667,
       133.33333333, 150.        ])

## **4. Design samples**

In [3]:
r_dist = Normal(loc=r_mean, scale=r_std)
s_dist = Normal(loc=s_mean, scale=s_std)
joint  = JointIndependent(marginals=[r_dist, s_dist])

x_pce_rvs = joint.rvs(n_samples)
x_val     = joint.rvs(n_samples_validation)

print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print(f"   Number of latent samples per design sample: {n_latent_samples}")
print(f"   Total simulations per time step: {(n_samples + n_samples_validation) * n_latent_samples}")

Samples generated successfully!
   Number of design samples: 2000
   Number of latent samples per design sample: 5000
   Total simulations per time step: 12500000


In [4]:
x_pce_rvs

array([[4.46342313, 1.48644209],
       [5.38899639, 2.69815247],
       [5.00561343, 1.25540539],
       ...,
       [3.56946321, 1.72103505],
       [4.03218509, 1.45313948],
       [4.77778412, 2.18480524]])

## **5. Generate the dataset at each time step**

Steps:

- $g$ evaluation;
- GLD fit; and
- saving `dataset_full`/`dataset_unique` for both splits.

In [5]:
print("="*60)
print("GENERATING THE BENCHMARK DATASET")
print("="*60)

generation_results = []
for t in times:
    result = generate_dataset_at_time_benchmark(
                                                   x_train=x_pce_rvs,
                                                   x_val=x_val,
                                                   time_step=t,
                                                   n_latent_samples=n_latent_samples,
                                                   k_factor_final=k_factor_final,
                                                   z1_std=z1_std,
                                                   z2_std=z2_std,
                                                   output_dir='.',
                                               )
    generation_results.append(result)

GENERATING THE BENCHMARK DATASET

----------------------------------------
GENERATING DATASET FOR TIME STEP: 0.0 years
----------------------------------------


KeyboardInterrupt: 

In [ ]:
with open('2500_dataset_full_train_0.0_benchmark.pkl', 'rb') as f:
    obj = dill.load(f)
obj.head()

,r,s,z1_latent,R_effective,z2_latent,S_effective,k factor,Time (years),g,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
0,3.401349,2.08612,0.951959,3.572999,1.123916,2.344624,1.0,0.0,1.228374,1.314791,6.569905,0.164401,0.120544,0.056415
1,3.401349,2.08612,1.001562,3.396043,0.952596,1.987231,1.0,0.0,1.408812,1.314791,6.569905,0.164401,0.120544,0.056415
2,3.401349,2.08612,1.053421,3.228859,0.915855,1.910585,1.0,0.0,1.318274,1.314791,6.569905,0.164401,0.120544,0.056415
3,3.401349,2.08612,1.024695,3.319377,0.833151,1.738053,1.0,0.0,1.581324,1.314791,6.569905,0.164401,0.120544,0.056415
4,3.401349,2.08612,0.998185,3.407535,0.901111,1.879826,1.0,0.0,1.527709,1.314791,6.569905,0.164401,0.120544,0.056415


## **6. Timing summary**

Cost of building the dataset, per time step.

In [ ]:
timing_rows = []
for result in generation_results:
    train_t = result['df_unique_train']['Processing time (s)']
    val_t   = result['df_unique_val']['Processing time (s)']
    timing_rows.append({
                           'Time (years)':   result['time_step'],
                           'n_train':        len(train_t),
                           'Train total (s)': train_t.sum(),
                           'Train mean (ms)': train_t.mean() * 1e3,
                           'n_val':          len(val_t),
                           'Val total (s)':  val_t.sum(),
                       })

emulator_timing = pd.DataFrame(timing_rows)

with open(f'{n_latent_samples}_emulator_timing_benchmark.pkl', 'wb') as f:
    dill.dump(emulator_timing, f)

train_total = emulator_timing['Train total (s)'].sum()
val_total   = emulator_timing['Val total (s)'].sum()

print(f"Train split - eg-value dataset generation time: {train_total:.1f} s")
print(f"Val split   - g-value dataset generation time: {val_total:.1f} s")
print(f"Total g-value dataset generation time (train + val): {train_total + val_total:.1f} s")
emulator_timing

Train split - eg-value dataset generation time: 23.4 s
Val split   - g-value dataset generation time: 11.6 s
Total g-value dataset generation time (train + val): 35.0 s


,Time (years),n_train,Train total (s),Train mean (ms),n_val,Val total (s)
0,0.000000,500,2.451832,4.903664,250,1.244647
1,16.666667,500,2.340988,4.681976,250,1.158463
2,33.333333,500,2.335934,4.671868,250,1.242482
3,50.000000,500,2.351744,4.703488,250,1.170401
4,66.666667,500,2.413446,4.826892,250,1.160200
5,83.333333,500,2.315978,4.631957,250,1.144871
6,100.000000,500,2.299619,4.599238,250,1.114061
7,116.666667,500,2.294267,4.588534,250,1.132866
8,133.333333,500,2.280119,4.560238,250,1.143264
9,150.000000,500,2.295608,4.591217,250,1.136969


## **7. Unique dataset statistics**

Concatenate the `dataset_unique` (train + val, all time steps) frames already held in `generation_results` and describe the columns, including the fitted lambdas.

In [ ]:
dataset_unique_all = pd.concat(
    [
        result[f'df_unique_{split}'].assign(split=split)
        for result in generation_results
        for split in ('train', 'val')
    ],
    ignore_index=True,
)

print(f"Combined unique dataset: {len(dataset_unique_all)} rows "
      f"({len(generation_results)} time steps x train/val splits)")
dataset_unique_all.describe()

Combined unique dataset: 7500 rows (10 time steps x train/val splits)


,r,s,lambda 1,lambda 2,lambda 3,lambda 4,Processing time (s)
count,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000,7500.000000
mean,5.012460,2.033968,0.348249,7.803138,0.135791,0.129752,0.004670
std,0.808967,0.639814,1.856628,4.592338,0.053384,0.052839,0.001877
min,2.859713,-0.060647,-4.534302,3.298130,-0.412679,-0.414634,0.003437
25%,4.429513,1.607262,-1.143183,5.816807,0.125617,0.119480,0.004277
50%,4.996460,2.051540,0.280706,6.865878,0.139965,0.133836,0.004746
75%,5.563735,2.459245,1.753081,8.464582,0.154001,0.147930,0.004967
max,7.697781,4.301989,5.886027,122.601964,0.220666,0.224072,0.092263
